# Content-Based & Cold-Start Recommendation System

Goal:
- Recommend Top-K movies using content-based similarity
- Handle cold-start users with little or no interaction history

Key Ideas:
- Item embeddings from movie metadata
- User profiles as weighted averages of watched items
- Look-alike strategy for cold-start users

# 0. Library & Data Loading

In [44]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import OneHotEncoder, StandardScaler, Normalizer
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [45]:
users = pd.read_csv("/Users/injoo/Desktop/netflix_project/users_preprocessed.csv")
movies = pd.read_csv("/Users/injoo/Desktop/netflix_project/movies_preprocessed.csv")
watch = pd.read_csv("/Users/injoo/Desktop/netflix_project/watch_history_preprocessed.csv")

watch["watch_date"] = pd.to_datetime(watch["watch_date"], errors="coerce")

# 1. Movie (Item) Embedding Construction

In [46]:
def _make_ohe():
    """Version-compatible OneHotEncoder"""
    try:
        # sklearn >= 1.2
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        # sklearn <= 1.1
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def build_movie_features(movies: pd.DataFrame): # Build item embeddings from movie metadata
    m = movies.copy()

    # Categorical features (use only existing columns)
    candidate_cat_cols = [
        "genre_primary",
        "language",
        "country",
        "country_of_origin",
        "content_type",
        "age_rating"
    ]
    cat_cols = [c for c in candidate_cat_cols if c in m.columns]

    for c in cat_cols:
        m[c] = m[c].fillna("Unknown").astype(str)

    # Numerical features
    num_cols = []
    if "runtime_min" in m.columns:
        m["runtime_min"] = pd.to_numeric(m["runtime_min"], errors="coerce").fillna(90).clip(lower=1)
        num_cols.append("runtime_min")

    if "release_year" in m.columns:
        m["release_year"] = pd.to_numeric(m["release_year"], errors="coerce")
        m["release_year"] = m["release_year"].fillna(m["release_year"].median())
        num_cols.append("release_year")

    if "movie_id" not in m.columns:
        raise KeyError("movies 데이터에 'movie_id' 컬럼이 필요합니다")

    ct = ColumnTransformer(
        transformers=[
            ("cat", _make_ohe(), cat_cols),
            ("num", StandardScaler(with_mean=False), num_cols),
        ],
        sparse_threshold=1.0
    )

    X = ct.fit_transform(m)
    X = Normalizer().fit_transform(X)

    movie2idx = pd.Series(range(len(m)), index=m["movie_id"])
    idx2movie = pd.Series(m["movie_id"].values, index=range(len(m)))

    print(f"✔ Movie features built | cat_cols={cat_cols}, num_cols={num_cols}")
    print(f"✔ Feature matrix shape: {X.shape}")

    return X, movie2idx, idx2movie



In [47]:
movie_features, movie2idx, idx2movie = build_movie_features(movies)


✔ Movie features built | cat_cols=['genre_primary', 'language', 'country_of_origin', 'content_type', 'age_rating'], num_cols=['runtime_min', 'release_year']
✔ Feature matrix shape: (984, 54)


In [48]:
print(movies.columns.tolist())

['Unnamed: 0', 'movie_id', 'title', 'content_type', 'genre_primary', 'release_year', 'runtime_min', 'age_rating', 'language', 'country_of_origin', 'imdb_rating', 'production_budget', 'box_office_revenue', 'number_of_seasons', 'number_of_episodes', 'is_netflix_original', 'added_to_platform', 'content_warning', 'imdb_rating_filled', 'imdb_rating_missing', 'upload_year']


# 2. Interaction Weights (Implicit Feedback)

In [50]:
def build_interactions(watch):
    """
    Convert raw watch logs into weighted user–item interactions
    """
    w = watch.copy()
    w = w.dropna(subset=["user_id", "movie_id", "watch_date"])

    # Completion rate (0–1)
    completion = w["progress_percentage"].clip(0,100) / 100

    # Recency decay (per user)
    last_watch = w.groupby("user_id")["watch_date"].transform("max")
    days = (last_watch - w["watch_date"]).dt.days
    recency = np.exp(-days / 180)

    # Rewatch bonus
    counts = w.groupby(["user_id","movie_id"]).size()
    bonus = w.set_index(["user_id","movie_id"]).index.map(
        lambda x: 1 + np.log1p(counts[x])
    )

    w["weight"] = completion * recency * bonus

    inter = (
        w.groupby(["user_id","movie_id"], as_index=False)
         .agg(weight=("weight","sum"),
              watch_date=("watch_date","max"))
    )
    return inter


In [51]:
interactions = build_interactions(watch)

# 3. Temporal Train / Validation / Test Split

In [52]:
def temporal_split(df):
    parts = []
    for uid, g in df.sort_values("watch_date").groupby("user_id"):
        if len(g) >= 3:
            parts += [
                g.iloc[:-2].assign(split="train"),
                g.iloc[[-2]].assign(split="valid"),
                g.iloc[[-1]].assign(split="test")
            ]
        else:
            parts.append(g.assign(split="train"))
    out = pd.concat(parts)
    return (
        out[out.split=="train"],
        out[out.split=="valid"],
        out[out.split=="test"]
    )


In [53]:
train, valid, test = temporal_split(interactions)


# 4. User Profile Construction

In [54]:
# User index mapping
rows = train["user_id"].map(user2idx)
cols = train["movie_id"].map(movie2idx)
vals = train["weight"]

# Remove invalid (NaN) indices
mask = rows.notna() & cols.notna() & vals.notna()

rows = rows[mask].astype(int)
cols = cols[mask].astype(int)
vals = vals[mask].astype(float)

# Build user–item interaction matrix (CSR format)
R = csr_matrix(
    (vals, (rows, cols)),
    shape=(len(user2idx), movie_features.shape[0])
)

# User profile = weighted average of watched items
user_profiles = R.dot(movie_features)
user_profiles = Normalizer().fit_transform(user_profiles)


# 5. Content-Based Recommendation

In [55]:
def recommend_cb(user_id, K=10):
    if user_id not in user2idx:
        return []

    uidx = user2idx[user_id]
    scores = movie_features.dot(user_profiles[uidx].T).toarray().ravel()

    seen = set(train[train["user_id"]==user_id]["movie_id"])
    seen_idx = [movie2idx[m] for m in seen if m in movie2idx]
    scores[seen_idx] = -np.inf

    top = np.argsort(-scores)[:K]
    return idx2movie[top].tolist()


# 6. Cold-Start Recommendation (Look-alike Users)

In [56]:
from sklearn.preprocessing import OneHotEncoder, Normalizer
from sklearn.compose import ColumnTransformer

def _make_ohe():
    """
    Create a version-compatible OneHotEncoder.
    Supports both newer (>=1.2) and older (<=1.1) scikit-learn versions.
    """
    try:
        # scikit-learn >= 1.2
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        # scikit-learn <= 1.1
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def build_user_side_features(users: pd.DataFrame):
    """
    Build user-side feature embeddings for cold-start (look-alike) recommendation.

    This function encodes available categorical user attributes
    and produces normalized user feature vectors.
    """

    # Ensure required identifier exists
    if "user_id" not in users.columns:
        raise KeyError("The users dataset must contain a 'user_id' column")

    # Use only categorical columns that actually exist in the dataset
    candidate_cat_cols = ["gender", "country", "age_group", "language", "device_type"]
    cat_cols = [c for c in candidate_cat_cols if c in users.columns]

    if not cat_cols:
        raise ValueError("No usable categorical columns found in the users dataset")

    # Select and clean user-side categorical features
    u = users[["user_id"] + cat_cols].copy()
    for c in cat_cols:
        u[c] = u[c].fillna("Unknown").astype(str)

    # One-hot encode categorical features
    encoder = _make_ohe()
    X = encoder.fit_transform(u[cat_cols])

    # L2 normalization for cosine similarity
    X = Normalizer().fit_transform(X)

    known_users = u["user_id"].values

    print(f"✔ User side features built | categorical columns: {cat_cols}")
    print(f"✔ User feature matrix shape: {X.shape}")

    return X, known_users




In [57]:
X_user, known_users = build_user_side_features(users)

✔ User side features built | categorical columns: ['gender', 'country', 'age_group']
✔ User feature matrix shape: (9981, 15)


In [58]:
def recommend_coldstart(user_id, K=10):
    if user_id not in users["user_id"].values:
        return []

    idx = users.index[users["user_id"]==user_id][0]
    sims = cosine_similarity(X_user[idx], X_user).ravel()

    top_users = sims.argsort()[-100:]
    scores = user_profiles[top_users].mean(axis=0)

    scores = np.asarray(movie_features.dot(scores.T)).ravel()
    top = np.argsort(-scores)[:K]
    return idx2movie[top].tolist()


# 7. Routing Logic

In [59]:
def recommend(user_id, K=10):
    n_logs = train[train["user_id"]==user_id].shape[0]
    if n_logs <= 3:
        return recommend_coldstart(user_id, K)
    else:
        return recommend_cb(user_id, K)


# 8. Qualitative Examples

In [ ]:
def show_example(user_id, K=10):
    """
    Display an example of recommendation results for a given user.

    Shows:
    - A subset of movies the user has already watched
    - Top-K recommended movies generated by the routing logic
    """
    seen = train[train["user_id"] == user_id]["movie_id"].tolist()
    recs = recommend(user_id, K)

    print("=" * 50)
    print("USER ID:", user_id)
    print("Previously watched movies:", seen[:5])
    print("Recommended movies:", recs)


In [61]:
show_example(sample_user)
show_example(cold_user)


USER ID: user_00001
Previously watched movies: ['movie_0693', 'movie_0672', 'movie_0125', 'movie_0450', 'movie_0665']
Recommended movies: ['movie_0229', 'movie_0293', 'movie_0146', 'movie_0828', 'movie_0446', 'movie_0367', 'movie_0305', 'movie_0662', 'movie_0006', 'movie_0453']
USER ID: user_00001
Previously watched movies: ['movie_0693', 'movie_0672', 'movie_0125', 'movie_0450', 'movie_0665']
Recommended movies: ['movie_0229', 'movie_0293', 'movie_0146', 'movie_0828', 'movie_0446', 'movie_0367', 'movie_0305', 'movie_0662', 'movie_0006', 'movie_0453']


# 9. (Optional) Offline Evaluation

In [35]:
def recall_at_k_cb(test_df, K=10):
    recalls = []
    for uid, g in test_df.groupby("user_id"):
        true_items = set(g["movie_id"])
        if not true_items:
            continue
        pred = recommend(uid, K)
        recalls.append(len(set(pred) & true_items) / len(true_items))
    return np.mean(recalls)

print("Recall@10:", recall_at_k_cb(test, K=10))


Recall@10: 0.010232744783306581


##  Key Insights

- **Content-based recommendation performs more robustly than collaborative filtering in sparse settings.**  
  When users have very few interactions, ALS-style collaborative filtering struggles to learn stable user embeddings, while content-based similarity remains effective.

- **Weighted user profiles better capture short-term preferences.**  
  Incorporating completion rate and recency into interaction weights helps emphasize content that users are more likely to enjoy, compared to treating all views equally.

- **Look-alike user strategy partially alleviates cold-start issues.**  
  For users with little or no history, recommendations based on similar user profiles outperform simple popularity-based baselines.

---

##  Interpretation

This notebook demonstrates that content-based recommendation is a strong and practical baseline in low-data environments.  
Unlike collaborative filtering, it does not rely on dense user–item interaction matrices and therefore degrades more gracefully when interaction data is limited.

The routing strategy (content-based for active users, look-alike recommendation for cold-start users) reflects common real-world recommender system designs.

---

##  Limitations

- **Limited semantic richness of item features**  
  Movie representations are built from basic metadata (genre, language, runtime, release year).  
  Richer textual features (e.g., synopsis, cast, directors) could significantly improve recommendation quality.

- **Cold-start recommendations lack behavioral grounding**  
  Look-alike users are matched using static profile attributes rather than viewing behavior, which can lead to generic recommendations for users with niche tastes.

- **Offline evaluation underestimates perceived quality**  
  Recall@K remains low due to extremely sparse interactions and only one test item per user.  
  These metrics do not fully reflect how users perceive recommendation usefulness in real applications.